# ML Assignment 2 - Digits Classification
Complete notebook that trains, evaluates and saves all models.

In [3]:
import os, joblib, pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report

In [4]:
data=load_digits()
X=pd.DataFrame(data.data,columns=[f'pixel_{i}' for i in range(data.data.shape[1])])
y=pd.Series(data.target,name='target')
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
os.makedirs('model',exist_ok=True)
print(X.shape,y.nunique())

(1797, 64) 10


In [5]:
models={
'Logistic Regression':Pipeline([('scaler',StandardScaler()),('classifier',LogisticRegression(max_iter=2000))]),
'Decision Tree':DecisionTreeClassifier(random_state=42),
'kNN':Pipeline([('scaler',StandardScaler()),('classifier',KNeighborsClassifier())]),
'Naive Bayes':GaussianNB(),
'Random Forest':RandomForestClassifier(random_state=42)}

In [6]:
results=[]
for name,model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    prob=model.predict_proba(X_test)
    auc=roc_auc_score(label_binarize(y_test,classes=range(10)),prob,multi_class='ovr',average='weighted')
    joblib.dump(model,f'model/{name.replace(" ","_")}.pkl')
    results.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'AUC':auc,'Precision':precision_score(y_test,pred,average='weighted'),'Recall':recall_score(y_test,pred,average='weighted'),'F1':f1_score(y_test,pred,average='weighted'),'MCC':matthews_corrcoef(y_test,pred)})
    print('\n'+name)
    print(confusion_matrix(y_test,pred))
    print(classification_report(y_test,pred))
results_df=pd.DataFrame(results).round(4)
results_df


Logistic Regression
[[36  0  0  0  0  0  0  0  0  0]
 [ 0 32  0  1  1  0  0  0  2  0]
 [ 0  0 35  0  0  0  0  0  0  0]
 [ 0  0  0 37  0  0  0  0  0  0]
 [ 0  0  0  0 36  0  0  0  0  0]
 [ 0  0  0  0  0 37  0  0  0  0]
 [ 0  0  0  0  0  0 35  0  1  0]
 [ 0  0  0  0  0  0  0 36  0  0]
 [ 0  4  0  0  0  0  0  0 31  0]
 [ 0  0  0  0  0  0  0  0  1 35]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       0.89      0.89      0.89        36
           2       1.00      1.00      1.00        35
           3       0.97      1.00      0.99        37
           4       0.97      1.00      0.99        36
           5       1.00      1.00      1.00        37
           6       1.00      0.97      0.99        36
           7       1.00      1.00      1.00        36
           8       0.89      0.89      0.89        35
           9       1.00      0.97      0.99        36

    accuracy                           0.97       36

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9722,0.9991,0.9724,0.9722,0.9722,0.9692
1,Decision Tree,0.8250,0.9028,0.8241,0.8250,0.8237,0.8057
2,kNN,0.9639,0.9951,0.9648,0.9639,0.9636,0.9600
3,Naive Bayes,0.8111,0.9707,0.8480,0.8111,0.8151,0.7940
4,Random Forest,0.9611,0.9992,0.9620,0.9611,0.9609,0.9569


In [7]:
test_data=X_test.copy()
test_data['target']=y_test.values
test_data.to_csv('test_data.csv',index=False)
results_df.to_csv('model_metrics.csv',index=False)
print('Artifacts created successfully.')

Artifacts created successfully.


## Generated files
- model/*.pkl
- test_data.csv
- model_metrics.csv